In [1]:
import os
from pathlib import Path
import jwst
print(jwst.__version__)
from jwst import datamodels
from jwst.datamodels import dqflags
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.lines import Line2D
from scipy.optimize import curve_fit, minimize
from scipy.stats import norm, poisson
from scipy.special import gamma
from pathlib import Path
# import natural units:
import natural_units as nu
# multi-core/thread:
import concurrent.futures
import math
import re

1.19.2


In [2]:
dn_min = -200
dn_max = 400
dn_range = range(dn_min, dn_max)
log_m_min  = -3
log_m_max  = 1
n_m    = 17   # From 1e-3  to 10 GeV
#log cs shift from the balloon line
cs_logshift_min = -6
cs_logshift_max = -3
cs_shift_numbers   = 25
m_grid      = np.logspace(log_m_min, log_m_max, n_m)
cs_logshift = np.linspace(cs_logshift_min, cs_logshift_max, cs_shift_numbers)

def gaussian(x, mean, stddev):
    return  np.exp(-((x - mean) ** 2) / (2 * stddev ** 2)) / stddev / np.sqrt(2*np.pi)
def poisson(x, lambda_param):
    return np.exp(-lambda_param) * np.power(lambda_param, x) / gamma(x+1)
def log_poisson(x, lambda_param):
    if x > 0:
        return x * np.log(lambda_param) - lambda_param - (x*np.log(x) - x + np.log(2*np.pi*x)/2 + 1/12/x)  # 这 lambda_param 不能是0。如果很小，结果很负，摆动很大，注意。log(N!) = N*log(N) - N + log(2pi N)/2 + 1/12/N - ...
    else:
        return -lambda_param

# fraction rescale for 0.1% and 0.05%
frac = '1'                   

output_dir = './advan_constraint/frac_'+frac+'/'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
likelihood_output_dir = output_dir + 'log_likelihood_grid/'
if not os.path.exists(likelihood_output_dir):
    os.makedirs(likelihood_output_dir)


binned_signal_path = Path('../Monte_Carlo_Signal_Simulation/results_w_lin/DM_binned_halo_'+frac)
binned_signals = [file for file in binned_signal_path.iterdir()]

signal_grid = np.loadtxt('../SHIELDING_RESULT/signal_grid_halo.txt')

In [3]:
best_constraints = ['jw01121130001_02102_00001_nrs2']

for input_file_base in best_constraints:
    filename = Path('../mask/results/masked/' + input_file_base + '.txt')

    fitting_output_dir = output_dir +'fitting_results/' + input_file_base
    if not os.path.exists(fitting_output_dir):
        os.makedirs(fitting_output_dir)
    pvd = np.loadtxt(filename)
    N_dn     = pvd[:,1]
    peak = np.argmax(N_dn) + dn_min
    dn_min_likelihood = peak - 50
    dn_max_likelihood = peak + 50 + 1 #难绷
    # background simulation
    # 样本数量
    total_count = np.sum(N_dn)

    def process_file(dm_poisson_file):
        file_stem = dm_poisson_file.stem
        dm_poisson = np.loadtxt(dm_poisson_file)

        i, j = file_stem.split('_')
        i = int(i)
        j = int(j)
        
        if os.path.exists(fitting_output_dir + '/fitting_' + str(i) + '_' + str(j) + '.txt'):
            print('Data point at ' + str(i) + '_' + str(j) + ' has already been fitted.')
        else:
            signal = signal_grid[cs_shift_numbers-1-j,i]
            if signal > peak:
                fitting_result = np.array([-1e10])
            else:
                # x[0]:peak_rescale  x[1]:peak_center  x[2]:read_noise
                def BKD_DM(x):
                    poisson_dn = np.zeros(dn_max-dn_min)
                    bkd_dm_analytical = np.zeros(dn_max-dn_min)
                    for dn in range(0, 100):
                        for dn_dm in range(0, dn+1):
                            poisson_dn[dn-dn_min] += dm_poisson[dn_dm-dn_min] * poisson(dn-dn_dm, x[1])
                    for dn in range(0, dn_max):
                        smear_min = dn_min
                        smear_max = dn_max
                        for dn_smear in range(smear_min, smear_max):
                            bkd_dm_analytical[dn_smear-dn_min] += poisson_dn[dn-dn_min] * ((1-x[3])*gaussian(dn_smear, dn, x[2])+x[3]*gaussian(dn_smear, dn, x[4]))
                    bkd_dm_analytical = bkd_dm_analytical * total_count * x[0]
                    log_likelihood_dm = 0
                    for k in range(dn_min_likelihood-dn_min,dn_max_likelihood-dn_min):
                        log_likelihood_dm += log_poisson(N_dn[k], bkd_dm_analytical[k])
                    return log_likelihood_dm, bkd_dm_analytical
                def log_likelihood_inverse(x):
                    return -BKD_DM(x)[0]
                # 初始猜测值
                initial_guess = [1, peak-signal, 13, 0.1, 18]
                bounds = [(0.9,1.1),(np.min((0.5*(peak-signal),0)),np.max((1.5*peak, 20))),(8,16),(0,1),(16,30)]
                # 使用 minimize 函数寻找最小值
                result = minimize(log_likelihood_inverse, initial_guess, bounds=bounds, method= 'SLSQP')
                fitting_result = np.append(result.x, -result.fun)
            np.savetxt(fitting_output_dir + '/fitting_' + file_stem + '.txt', fitting_result)

    def process_files_in_parallel(file_list):
        with concurrent.futures.ProcessPoolExecutor() as executor:
            executor.map(process_file, file_list)
    process_files_in_parallel(binned_signals)

    # Calculate the log likelihood of null hypothesis
    def log_likelihood_bkd_inverse(x):
        bkd_analytical = np.zeros(dn_max-dn_min)
        for dn in range(0, 100):
            poisson_dn = poisson(dn, x[1])
            smear_min = dn_min
            smear_max = dn_max
            for dn_smear in range(smear_min, smear_max):
                bkd_analytical[dn_smear-dn_min] += poisson_dn * ((1-x[3])*gaussian(dn_smear, dn, x[2])+x[3]*gaussian(dn_smear, dn, x[4]))
        bkd_analytical = bkd_analytical * total_count * x[0]
        log_likelihood_0 = 0
        for k in range(dn_min_likelihood-dn_min,dn_max_likelihood-dn_min):
            log_likelihood_0 += log_poisson(N_dn[k], bkd_analytical[k])
        return -log_likelihood_0
    # 初始猜测值
    initial_guess_bkd = [1, peak, 13, 0.1, 18]
    bounds_bkd = [(0.9,1.1),(0,np.max((1.5*peak, 20))),(8,16),(0,1),(16,30)]
    # 使用 minimize 函数寻找最小值
    result_bkd = minimize(log_likelihood_bkd_inverse, initial_guess_bkd, bounds=bounds_bkd, method= 'SLSQP')

    log_likelihood_grid = np.zeros((n_m, cs_shift_numbers))
    for i in range(n_m):
        for j in range(cs_shift_numbers):
            log_likelihood_grid[i,j] = np.atleast_1d(np.loadtxt(fitting_output_dir + '/fitting_' + str(i) + '_' + str(j) + '.txt'))[-1]

    np.savetxt(likelihood_output_dir + input_file_base +'_log_likelihood_grid.txt', log_likelihood_grid, header = str(-result_bkd.fun), comments='')